In [4]:
import pandas as pd
import numpy as np
from time import perf_counter
from datasets import load_dataset
from memory_profiler import memory_usage
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, precision_recall_fscore_support
from transformers import DistilBertTokenizer, DistilBertModel


In [5]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRETRAINED_MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5 

print(f"Using device: {DEVICE}")

Using device: cuda


In [6]:
ds = load_dataset("cardiffnlp/tweet_eval", "sentiment")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,text,label
0,"""QT @user In the original draft of the 7th boo...",2
1,"""Ben Smith / Smith (concussion) remains out of...",1
2,Sorry bout the stream last night I crashed out...,1
3,Chase Headley's RBI double in the 8th inning o...,1
4,@user Alciato: Bee will invest 150 million in ...,2
...,...,...
45610,"@user \""""So amazing to have the beautiful Lady...",2
45611,"9 September has arrived, which means Apple's n...",2
45612,Leeds 1-1 Sheff Wed. Giuseppe Bellusci securin...,2
45613,@user no I'm in hilton head till the 8th lol g...,1


In [7]:
class MultiClassClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels  # Labels should be integers: 0, 1, 2, ..., num_classes-1
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)  # Changed to scalar tensor of type long
        }

In [8]:
class DistilBertForMultiClassClassification(nn.Module):
    def __init__(self, num_classes):
        super(DistilBertForMultiClassClassification, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained(PRETRAINED_MODEL_NAME)
        self.pre_classifier = nn.Linear(768, 768)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_classes)  # Output size is num_classes
    
    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden_state = outputs[0][:, 0]  # CLS token
        pooled_output = self.pre_classifier(hidden_state)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits  # Return raw logits, no sigmoid

In [9]:
def get_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    precisions, recalls, f1s, supports = precision_recall_fscore_support(y_true, y_pred)

    return acc, precisions, recalls, f1s

In [10]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, scheduler=None, epochs=EPOCHS):
    best_val_loss = float('inf')
    start_train = perf_counter()
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_true = []
        
        progress_bar = tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{epochs}', leave=False)
        for batch in progress_bar:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size,)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)  # Shape: (batch_size, num_classes)
            loss = criterion(outputs, labels)  # CrossEntropyLoss expects logits and long labels
            train_loss += loss.item()
            
            # Accumulate predictions and true labels for metrics
            preds = torch.argmax(outputs, dim=1).cpu().numpy()  # Get class indices
            train_preds.extend(preds)
            train_true.extend(labels.cpu().numpy())
            
            loss.backward()
            optimizer.step()
            progress_bar.set_postfix({'loss': loss.item()})
        
        if scheduler:
            scheduler.step()
            
        train_loss /= len(train_dataloader)
        train_preds = np.array(train_preds)
        train_true = np.array(train_true)
        
        train_acc, train_precisions, train_recalls, train_f1s = get_metrics(train_true, train_preds)
        
        start_val = perf_counter()
        model.eval()
        val_loss = 0
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch in tqdm(val_dataloader, desc="Validation", leave=False):
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(labels.cpu().numpy())
        
        val_time = perf_counter() - start_val
        
        val_loss /= len(val_dataloader)
        val_preds = np.array(val_preds)
        val_true = np.array(val_true)
        
        val_acc, val_precisions, val_recalls, val_f1s = get_metrics(val_true, val_preds)
        
        print(f"Epoch {epoch + 1}/{epochs} - Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1s}, Prec: {train_precisions}, Recall: {train_recalls}")
        print(f"Epoch {epoch + 1}/{epochs} - Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1s}, Prec: {val_precisions}, Recall: {val_recalls}, Val Time: {val_time:.2f} sec")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'results/bert_multiclass1.pt')
            print("Model saved!")
    
    total_train_time = perf_counter() - start_train
    print(f"Total Training Time: {total_train_time:.2f} seconds")
    
    return train_acc, train_precisions, train_recalls, train_f1s, val_acc, val_precisions, val_recalls, val_f1s, total_train_time, val_time

In [11]:
def evaluate_model(model, test_dataloader):
    model.eval()
    predictions = []
    true_labels = []
    classification_times = []
    
    start_test = perf_counter()
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            for i in range(input_ids.size(0)):
                input_id = input_ids[i].unsqueeze(0)
                attention_mask_sample = attention_mask[i].unsqueeze(0)
                label = labels[i].item()
                
                start_time = perf_counter()
                
                output = model(input_ids=input_id, attention_mask=attention_mask_sample)  # Shape: (1, num_classes)
                pred = torch.argmax(output, dim=1).item()  # Scalar integer
                
                predictions.append(pred)
                true_labels.append(label)
                classification_times.append(perf_counter() - start_time)
    
    total_test_time = perf_counter() - start_test
    print(f"Test Time: {total_test_time:.2f} seconds")
    
    predictions = np.array(predictions)
    true_labels = np.array(true_labels)
    
    acc, precisions, recalls, f1s = get_metrics(true_labels, predictions)
    
    print("Test Metrics:")
    print("Accuracy:", acc)
    print("F1s:", f1s)
    print("Precisions:", precisions)
    print("Recalls:", recalls)
    
    return predictions, true_labels

In [12]:
train_texts = train_df['text'].values
train_labels = train_df['label'].values  # Must be integers: 0, 1, 2, ..., num_classes-1

val_texts = val_df['text'].values
val_labels = val_df['label'].values

test_texts = test_df['text'].values
test_labels = test_df['label'].values

tokenizer = DistilBertTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)

# Use MultiClassClassificationDataset instead of BinaryClassificationDataset
train_dataset = MultiClassClassificationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = MultiClassClassificationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = MultiClassClassificationDataset(test_texts, test_labels, tokenizer, MAX_LEN)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

seeds = [2,3,5]

num_classes = train_df['label'].nunique()

avg_train_acc = 0
avg_train_precs = np.zeros(num_classes)
avg_train_recalls = np.zeros(num_classes)
avg_train_f1s = np.zeros(num_classes)
avg_max_memory_usage_train = 0
avg_max_vram_usage_train = 0
avg_total_train_time = 0

avg_val_acc = 0
avg_val_precs = np.zeros(num_classes)
avg_val_recalls = np.zeros(num_classes)
avg_val_f1s = np.zeros(num_classes)
avg_total_val_time = 0

avg_test_acc = 0
avg_test_precs = np.zeros(num_classes)
avg_test_recalls = np.zeros(num_classes)
avg_test_f1s = np.zeros(num_classes)
avg_max_memory_usage_test = 0
avg_max_vram_usage_test = 0
avg_total_test_time = 0

for seed in seeds:
    torch.manual_seed(seed)
    # Use multi-class model with num_classes parameter
    model = DistilBertForMultiClassClassification(num_classes)
    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    # Use CrossEntropyLoss for multi-class
    criterion = nn.CrossEntropyLoss()

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    max_memory_usage_train, retval = memory_usage(
        (train_model, (model, train_dataloader, val_dataloader, optimizer, criterion), {'epochs': EPOCHS}),
        max_usage=True,
        retval=True
    )

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    (train_acc, train_precisions, train_recalls, train_f1s, 
     val_acc, val_precisions, val_recalls, val_f1s, 
     total_train_time, val_time) = retval

    # Load the best model saved during training (ensure train_model saves to this path)
    model.load_state_dict(torch.load('results/bert_multiclass1.pt'))

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start = perf_counter()
    max_memory_usage_test, retval = memory_usage(
        (evaluate_model, (model, test_dataloader), {}),
        max_usage=True,
        retval=True
    )
    total_time_test = perf_counter() - start

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    predictions, true_labels = retval

    test_acc, test_precisions, test_recalls, test_f1s = get_metrics(true_labels, predictions)

    avg_train_acc += train_acc
    avg_train_precs += train_precisions
    avg_train_recalls += train_recalls
    avg_train_f1s += train_f1s
    avg_max_memory_usage_train += max_memory_usage_train
    avg_max_vram_usage_train += max_vram_usage_train
    avg_total_train_time += total_train_time

    avg_val_acc += val_acc
    avg_val_precs += val_precisions
    avg_val_recalls += val_recalls
    avg_val_f1s += val_f1s
    avg_total_val_time += val_time

    avg_test_acc += test_acc
    avg_test_precs += test_precisions
    avg_test_recalls += test_recalls
    avg_test_f1s += test_f1s
    avg_max_memory_usage_test += max_memory_usage_test
    avg_max_vram_usage_test += max_vram_usage_test
    avg_total_test_time += total_time_test

avg_train_acc /= len(seeds)
avg_train_precs /= len(seeds)
avg_train_recalls /= len(seeds)
avg_train_f1s /= len(seeds)
avg_max_memory_usage_train /= len(seeds)
avg_max_vram_usage_train /= len(seeds)
avg_total_train_time /= len(seeds)

avg_val_acc /= len(seeds)
avg_val_precs /= len(seeds)
avg_val_recalls /= len(seeds)
avg_val_f1s /= len(seeds)
avg_total_val_time /= len(seeds)

avg_test_acc /= len(seeds)
avg_test_precs /= len(seeds)
avg_test_recalls /= len(seeds)
avg_test_f1s /= len(seeds)
avg_max_memory_usage_test /= len(seeds)
avg_max_vram_usage_test /= len(seeds)
avg_total_test_time /= len(seeds)

avg_classification_time = avg_total_test_time / len(test_texts)

avg_classification_time

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/3 - Train Loss: 0.6675, Acc: 0.6970, F1: [0.59140666 0.70304073 0.7301998 ], Prec: [0.62673611 0.67676677 0.74988191], Recall: [0.55984774 0.73143714 0.71152446]
Epoch 1/3 - Val Loss: 0.6276, Acc: 0.7190, F1: [0.64516129 0.6683871  0.79447323], Prec: [0.57356608 0.76064611 0.75163399], Recall: [0.73717949 0.59608746 0.84249084], Val Time: 3.85 sec
Model saved!


Epoch 2/3 - Train Loss: 0.5047, Acc: 0.7819, F1: [0.72920683 0.77499225 0.8109915 ], Prec: [0.73864623 0.76447059 0.82023953], Recall: [0.72000564 0.78580758 0.80194969]
Epoch 2/3 - Val Loss: 0.6233, Acc: 0.7330, F1: [0.6130137  0.72216097 0.78901373], Prec: [0.65808824 0.69312169 0.80715198], Recall: [0.57371795 0.75373993 0.77167277], Val Time: 3.87 sec
Model saved!


C:\Users\Rafael\AppData\Local\Temp\ipykernel_15900\99819750.py:73: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('results/bert_multiclass1.p

Epoch 3/3 - Train Loss: 0.3391, Acc: 0.8610, F1: [0.84514027 0.85174552 0.8781158 ], Prec: [0.84935213 0.84963419 0.87892905], Recall: [0.84096997 0.85386736 0.87730405]
Epoch 3/3 - Val Loss: 0.6634, Acc: 0.7375, F1: [0.63837012 0.71362049 0.79621973], Prec: [0.67870036 0.72202591 0.77116705], Recall: [0.6025641  0.70540852 0.82295482], Val Time: 3.86 sec
Total Training Time: 834.11 seconds


...

C:\Users\Rafael\AppData\Local\Temp\ipykernel_15900\99819750.py:73: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('results/bert_multiclass1.p

Epoch 3/3 - Train Loss: 0.3411, Acc: 0.8605, F1: [0.84389316 0.85173654 0.8772186 ], Prec: [0.84371477 0.84995183 0.87944141], Recall: [0.84407162 0.85352876 0.875007  ]
Epoch 3/3 - Val Loss: 0.7466, Acc: 0.7140, F1: [0.61884669 0.68190018 0.78843788], Prec: [0.55137845 0.71410579 0.79429988], Recall: [0.70512821 0.65247411 0.78266178], Val Time: 6.85 sec
Total Training Time: 937.72 seconds


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
Testing: 100%|██████████| 768/768 [01:10<00:00, 10.83it/s]


Test Time: 70.90 seconds
Test Metrics:
Accuracy: 0.6928524910452621
F1s: [0.71281043 0.68606887 0.67588141]
Precisions: [0.69654012 0.71298819 0.64463126]
Recalls: [0.72985901 0.6611083  0.71031579]


0.005557822506240897

In [13]:
# save results to txt
with open("results/bert_multiclass1.txt", "w") as f:
    f.write(f"Average Train Accuracy: {avg_train_acc}\n")
    f.write(f"Average Train Precisions: {avg_train_precs}\n")
    f.write(f"Average Train Recalls: {avg_train_recalls}\n")
    f.write(f"Average Train F1s: {avg_train_f1s}\n")
    f.write(f"Average Max Memory Usage Train: {avg_max_memory_usage_train}\n")
    f.write(f"Average Max VRAM Usage Train: {avg_max_vram_usage_train}\n")
    f.write(f"Average Total Train Time: {avg_total_train_time}\n")
    f.write("\n")
    f.write(f"Average Val Accuracy: {avg_val_acc}\n")
    f.write(f"Average Val Precisions: {avg_val_precs}\n")
    f.write(f"Average Val Recalls: {avg_val_recalls}\n")
    f.write(f"Average Val F1s: {avg_val_f1s}\n")
    f.write(f"Average Total Val Time: {avg_total_val_time}\n")
    f.write("\n")
    f.write(f"Average Test Accuracy: {avg_test_acc}\n")
    f.write(f"Average Test Precisions: {avg_test_precs}\n")
    f.write(f"Average Test Recalls: {avg_test_recalls}\n")
    f.write(f"Average Test F1s: {avg_test_f1s}\n")
    f.write(f"Average Max Memory Usage Test: {avg_max_memory_usage_test}\n")
    f.write(f"Average Max VRAM Usage Test: {avg_max_vram_usage_test}\n")
    f.write(f"Average Total Test Time: {avg_total_test_time}\n")
    f.write("\n")
    f.write(f"Average Classification Time: {avg_classification_time}\n")
    f.write(f"Lines classified {len(test_texts)}\n")

    f.close()